# T04 — Formal hook and pooling validation

Run this notebook from the repository root on a CUDA GPU. It uses the canonical T03 modules and writes the two required T04 JSON reports.


In [ ]:
%pip install -q -U transformers accelerate huggingface_hub pyyaml pytest


## 1. Set the repository root


In [ ]:
from pathlib import Path
import os

# Change this path if the repository is mounted elsewhere.
REPO_ROOT = Path.cwd()
os.chdir(REPO_ROOT)
print('Repository root:', REPO_ROOT)

required = [
    'configs/method_frozen.yaml',
    'src/config.py',
    'src/generation.py',
    'src/activation_extraction.py',
    'src/runtime_hooks.py',
    'tools/run_t04_hook_and_pooling_validation.py',
    'tests/test_hook_and_pooling.py',
]
for path in required:
    print(path, 'OK' if Path(path).exists() else 'MISSING')


## 2. Run the CPU-only tests first


In [ ]:
!pytest -q tests/test_hook_and_pooling.py


## 3. Run the short GPU validation


In [ ]:
!python tools/run_t04_hook_and_pooling_validation.py \
  --config configs/method_frozen.yaml \
  --dtype bfloat16 \
  --attn-implementation eager \
  --max-new-tokens 128


## 4. Inspect the reports


In [ ]:
import json
from pathlib import Path

for path in [
    Path('results/validation/hook_validation_v3.json'),
    Path('results/validation/pooling_validation_v3.json'),
]:
    print('\n' + '=' * 80)
    print(path)
    data = json.loads(path.read_text())
    print('STATUS:', data['status'])
    print(json.dumps(data['checks'], indent=2))


## 5. Run the complete test file again


In [ ]:
!pytest -q tests/test_hook_and_pooling.py
